# Tiny ImageNet-C sơ bộ trên Kaggle

Dùng Dataset hiện có: 200 lớp, JPEG 64×64, tên `test_*.JPEG`. **1 panel, 3 kịch bản, W=0/4/16, source/norm/Tent/SAR, reset none/all: 72 cặp, 144 lượt Q.** Prefix=512, washout tối đa=512, Q=512, batch size=32.

Mô hình: ResNet50 ImageNet-1K V1 pretrained, chỉ giữ 200 logits tương ứng các synset của Dataset **trước** khi tính softmax, entropy và loss. Dùng transform của trọng số ImageNet (resize 256, crop 224). Không huấn luyện mô hình nguồn Tiny; kết quả là exploratory transfer trên Tiny, không phải kết quả Stage B ImageNet-C hay baseline Tiny đã được huấn luyện.

**Trước Run All:** bật GPU + Internet; Add Input Dataset nguồn mới từ `streaming-tta-kaggle-tiny-source.zip`; giữ Dataset ảnh hiện có. Có thể giữ Input nguồn cũ: notebook chỉ chọn nguồn chứa `scripts/tiny_imagenetc.py`. Xem `research/kaggle_tiny_preliminary.md`.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path
import shutil, subprocess, sys, json, zipfile, tempfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "streaming-tta-tiny-state-memory"
sources = [
    p.parent.parent for p in INPUT.rglob("tiny_imagenetc.py")
    if p.parent.name == "scripts"
    and (p.parent.parent / "src/historytta").is_dir()
    and (p.parent.parent / "configs/kaggle_tiny_preliminary.yaml").is_file()
]
if not REPO.exists():
    if len(sources) == 1:
        shutil.copytree(sources[0], REPO)
    elif not sources:
        bundles = list(INPUT.rglob("streaming-tta-kaggle-tiny-source.zip"))
        if len(bundles) != 1:
            raise RuntimeError("Add Input the NEW source bundle streaming-tta-kaggle-tiny-source.zip.")
        with tempfile.TemporaryDirectory(prefix="tiny_source_", dir=WORK) as temp:
            temp = Path(temp)
            with zipfile.ZipFile(bundles[0]) as archive:
                if sum(i.file_size for i in archive.infolist()) > 100_000_000:
                    raise RuntimeError("Unexpectedly large source bundle.")
                expected_root = temp / "streaming-tta-state-memory"
                for info in archive.infolist():
                    if not (temp / info.filename).resolve().is_relative_to(expected_root.resolve()):
                        raise RuntimeError("Unexpected ZIP path: " + info.filename)
                    if (info.external_attr >> 16) & 0o170000 == 0o120000:
                        raise RuntimeError("Source symlinks are not supported.")
                archive.extractall(temp)
            shutil.copytree(expected_root, REPO)
    else:
        raise RuntimeError("Multiple NEW source Datasets found; keep exactly one.")
if not (REPO / "scripts/tiny_imagenetc.py").is_file():
    raise RuntimeError("Tiny source is missing. Attach the new bundle and use a fresh session.")
os.chdir(REPO)
print("Tiny repository:", REPO)


## 1. Tự tìm dữ liệu

Nguồn mặc định là Dataset bạn đang dùng. Nếu đổi Dataset, sửa `DATA_SOURCE_URL`. Không sửa batch size, W hay các counts để bỏ qua lỗi. Sampler giữ hash split pilot/reserve, không lấy ảnh reserve khi thiếu pilot.

In [ ]:
DATA_SOURCE_URL = "https://www.kaggle.com/datasets/husnifdu/imagenet-c"
domains = ["gaussian_noise", "brightness", "defocus_blur"]
roots = [
    p.parent for p in INPUT.rglob("gaussian_noise")
    if all((p.parent / d / "5").is_dir() for d in domains)
]
if len(roots) != 1:
    print("Candidate image roots:", roots)
    raise RuntimeError("Attach one Tiny ImageNet-C Dataset containing all three corruption/5 directories.")
DATA_ROOT = roots[0]
CONFIG = REPO / "configs/kaggle_tiny_preliminary.yaml"
CLASS_INDEX = REPO / "datasets/metadata/imagenet_class_index.json"
DOWNLOAD_CLASS_INDEX = True
MAX_HOURS = 10.0
EXPORT_DIR = WORK / "kaggle_tiny_exports"

print("DATA_ROOT:", DATA_ROOT)
for domain in domains:
    variant = DATA_ROOT / domain / "5"
    count = sum(p.is_dir() for p in variant.iterdir())
    print(domain, "classes:", count)
    if count != 200:
        raise RuntimeError("This notebook requires exactly 200 Tiny classes in every variant.")


## 2. Kiểm tra môi trường

Giữ Torch/torchvision sẵn có của Kaggle. Internet dùng để lấy mapping ImageNet 35 KB, trọng số ResNet50 V1 và gói nhỏ còn thiếu. Notebook không tải archive ảnh.

In [ ]:
import importlib.util
for module in ["torch", "torchvision"]:
    if importlib.util.find_spec(module) is None:
        raise RuntimeError("Use a Kaggle GPU image with " + module)
packages = {"yaml": "PyYAML", "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
            "matplotlib": "matplotlib", "PIL": "Pillow", "pytest": "pytest", "nbformat": "nbformat"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import torch, torchvision
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle Settings.")
print("Torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda, "GPU:", torch.cuda.get_device_name(0))
probe = torch.ones((8, 8), device="cuda")
print("CUDA smoke:", (probe @ probe).mean().item())
del probe
torch.cuda.empty_cache()


## 3. Kế hoạch và tests nhỏ

Tests dùng fixtures CPU, không tải trọng số ResNet50. Đây là kiểm tra code, không phải kết quả Tiny thật.

In [ ]:
def helper(*arguments):
    subprocess.run([sys.executable, "-u", "scripts/kaggle_preliminary.py",
                    *map(str, arguments), "--config", str(CONFIG)], cwd=REPO, check=True)

helper("plan")
subprocess.run([sys.executable, "-m", "pytest", "-q",
                "tests/test_tiny_imagenetc.py", "tests/test_adapters.py",
                "tests/test_protocol.py", "tests/test_evaluate.py"], cwd=REPO, check=True)


## 4. Tạo manifest Tiny

Mapping nhãn 0..199 được tạo theo thứ tự synset và ánh xạ sang chỉ số ImageNet gốc. Ba corruption phải cùng ID và lớp. JPEG được chọn phải 64×64. Cần **1536 ID pilot**, tương ứng 4608 file JPEG qua ba corruption. Mirror được ghi rõ nguồn khai báo và hash ảnh được chọn; không tuyên bố đã kiểm MD5 archive chính thức.

In [ ]:
arguments = ["prepare", "--data-root", DATA_ROOT, "--source-url", DATA_SOURCE_URL,
             "--class-index", CLASS_INDEX]
if DOWNLOAD_CLASS_INDEX:
    arguments.append("--download-class-index")
helper(*arguments)


## 5. Chạy, audit và xuất ZIP

Mọi adaptation loss được tính trên 200 logits. SAR margin=0.4×log(200). Runner chỉ dùng GPU đầu tiên. Cap subprocess 10 giờ không bao gồm chuẩn bị dữ liệu, tests hoặc render hình.

Kết quả hoàn chỉnh cần run_status=complete, audit passed và tất cả controls đạt. `finally` xuất diagnostics cả khi runner lỗi; nếu Kaggle dừng cả session thì xuất ZIP không được bảo đảm. Không tự sửa trạng thái thành complete. Để chạy lại, đổi cả experiment_id/output; để export lại chọn thư mục EXPORT_DIR mới.

In [ ]:
try:
    helper("run", "--max-hours", MAX_HOURS)
finally:
    helper("export", "--export-dir", EXPORT_DIR)

print("Download from Kaggle Output:", EXPORT_DIR)


## 6. Đọc W16 và accuracy

Q=512: một quyết định khác tương ứng 0.1953125 điểm phần trăm. Giữ từng kịch bản riêng khi xem bảng. Disagreement không tự biểu thị lợi/hại, và bằng 0 không đảm bảo logits/loss giống nhau.

Hai CSV chính nằm trong results: `preliminary_history.csv`, `preliminary_performance.csv`. ZIP gồm source anchor, mapping/projected indices, NPZ, manifests, logs, bảng/hình; không gồm ảnh Input. Kết quả chỉ thuộc **Tiny exploratory transfer**, không so trực tiếp với số Stage A/Stage B ImageNet-C.

In [ ]:
import pandas as pd
import yaml
from IPython.display import display, FileLink

cfg = yaml.safe_load(CONFIG.read_text())
RESULTS = REPO / cfg["output"]
status = json.loads((RESULTS / "run_status.json").read_text())
audit = json.loads((RESULTS / "evaluation_audit.json").read_text())
controls = json.loads((RESULTS / "controls.json").read_text())
assert status["status"] == "complete" and audit["passed"]
assert controls and all(c["passed"] for c in controls)
print("Run:", status)
print("Audit:", audit["counts"])
history = pd.read_csv(RESULTS / "preliminary_history.csv")
display(history[history.washout_batches == 16])
performance = pd.read_csv(RESULTS / "preliminary_performance.csv")
display(performance[(performance.washout_batches == 16) & (performance.intervention == "none")])
archive = EXPORT_DIR / (cfg["experiment_id"] + "_artifacts.zip")
print("ZIP:", archive)
display(FileLink(str(archive)))
